<div align="center">

# Modelo de Churn / Attrition Bancario

**Panel temporal CIF × mes → feature engineering → modelos → ensemble → explicabilidad**

</div>

---

## Cómo usarlo

Corre las celdas de **Setup** (instalan dependencias, montan Drive y traen el código),
elige la fuente de datos y ejecuta el resto en orden. El pipeline es idéntico en ambos
casos: lo único que cambia es de dónde salen las tablas.

| `data.source` | Qué hace | Cuándo usarlo |
|:---|:---|:---|
| `synthetic` | Genera datos simulados en memoria | **Empieza aquí.** Valida el pipeline sin tocar datos reales |
| `drive` | Lee los extractos del agente SQL desde Google Drive | Cuando tengas las tablas cargadas |

## Dónde van los datos reales

Todo cuelga de `MyDrive/churn_model/data/raw/` — la ruta se configura en `data.drive_dir`
y los nombres de archivo en `data.files`.

| Query | Archivo | Contenido |
|:---|:---|:---|
| Q1 | `q1_party_master.csv` | Maestro de clientes — un registro por CIF |
| Q2 | `q2_cif_account_relationship.csv` | Relación CIF ↔ cuenta y rol (primary, joint, signer…) |
| Q3 | `q3_deposit_monthly.csv` | Saldo de depósitos a fin de mes |
| Q4 | `q4_loan_monthly.csv` | Préstamos vigentes por mes |
| Q5 | `q5_txn_monthly.csv` | Actividad transaccional mensual |
| Q6 | `q6_digital_contact_monthly.csv` | Uso de canales digitales y contactos |
| Q7 | `q7_product_events.csv` | Aperturas, cierres y eventos de producto |

> **Q4–Q7 son opcionales** (`data.optional`). Puedes empezar a modelar con Q1–Q3
> mientras llegan las demás; el pipeline tolera su ausencia.

## 0 · Setup

In [ ]:
# Colab ya trae numpy/pandas/sklearn/torch/xgboost
!pip install -q lightgbm catboost shap pyyaml 2>/dev/null
print("✓ dependencias listas")

In [ ]:
# ── Montar Google Drive ──
# Fuera de Colab esto no hace nada, así que la misma celda sirve en local.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✓ Drive montado")
else:
    print("ℹ️  Fuera de Colab — se omite el montaje")

In [ ]:
# ── Traer el código ──
# En Colab clona el repo. En local usa la carpeta actual.
# Si prefieres tener el código en Drive, apunta PROJECT_DIR ahí y omite el clone.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DanielRegaladoUMiami/bank-churn-attrition-design.git"

if IN_COLAB:
    PROJECT_DIR = Path("/content/churn")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"✓ Proyecto: {PROJECT_DIR}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10})

from churn.config import Config
from churn.data import load_tables, profile, validate
from churn.panel import build_cif_month_state, label_attrition_state, build_panel
from churn.features import build_features, select_feature_columns, clean_features
from churn.evaluation import (temporal_split, evaluate, summarize, decile_table,
                              calibration_table, calibrate, make_cv)
from churn.models import tune_all, build_stack, build_sequences, SequenceModel, SEQUENCE_COLS
from churn.explain import (shap_analysis, global_importance, explain_customer,
                           counterfactual_analysis, sanity_check)

cfg = Config.load("config.yaml")

# ⬇️ CAMBIA ESTO cuando tengas los extractos reales en Drive
cfg.data["source"] = "synthetic"      # "synthetic" | "drive"

print(f"Fuente de datos: {cfg.data['source']}")
print(f"Ventanas: feature={cfg.panel['feature_window_months']}m  "
      f"gap={cfg.panel['gap_months']}m  "
      f"performance={cfg.panel['performance_window_months']}m")

## 1 · Carga y validación

`load_tables` devuelve las 7 tablas con el mismo esquema, vengan de donde vengan.
`validate` las contrasta contra el contrato declarado en `churn/config.py`:

- llaves duplicadas,
- columnas obligatorias faltantes,
- nulos por encima del umbral tolerado.

In [ ]:
tables = load_tables(cfg)
profile(tables)

In [ ]:
_ = validate(tables)

## 2 · EDA

Tres cosas que hay que mirar **antes** de modelar. Ninguna es decorativa: cada una
fija un parámetro concreto del pipeline.

| Gráfico | Qué decide |
|:---|:---|
| Aperturas y cierres por mes | Si hubo una conversión de core, y dónde cortar la historia |
| Distribución de saldos | El umbral de saldo bajo del target (`target.low_balance_abs`) |
| Mix de productos y roles | Cuánto pesan las cuentas joint y los authorized signers |

> **Un pico aislado de cierres casi nunca es comportamiento de clientes.** Suele ser
> una migración de core. Si aparece, corta la historia ahí en lugar de dejar que el
> modelo aprenda el artefacto.

In [ ]:
# ── 1. Aperturas vs cierres por mes: detector de conversión de core ──
ev = tables["events"]
opens = (ev[ev.event_type == "account_open"].groupby(ev.event_date.dt.to_period("M"))
         .size().rename("aperturas"))
closes = (ev[ev.event_type == "account_close"].groupby(ev.event_date.dt.to_period("M"))
          .size().rename("cierres"))
flow = pd.concat([opens, closes], axis=1).fillna(0)
flow = flow[flow.index >= tables["deposits"].as_of_month.min().to_period("M")]

fig, ax = plt.subplots(figsize=(12, 4))
flow.plot(ax=ax, marker="o", ms=3)
ax.set_title("Aperturas y cierres por mes — un pico aislado = posible conversión de core")
ax.set_xlabel(""); ax.legend()
plt.tight_layout(); plt.show()

z = (flow.cierres - flow.cierres.mean()) / flow.cierres.std()
if (z.abs() > 3).any():
    print(f"⚠️  Meses con cierres anómalos (|z|>3): {list(flow.index[z.abs() > 3])}")
else:
    print("✓ Sin picos anómalos de cierres")

In [ ]:
# ── 2. Distribución de saldos: de aquí sale el umbral del target ──
dep = tables["deposits"]
bal = dep.loc[dep.eom_balance > 0, "eom_balance"]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(np.log10(bal.clip(lower=1)), bins=60, color="steelblue")
axes[0].set_title("log10(saldo) — cuentas con saldo > 0"); axes[0].set_xlabel("log10 USD")
for p in (0.5, 1, 2, 3):
    axes[0].axvline(p, color="crimson", ls="--", lw=0.8)

qs = bal.quantile([.05, .10, .25, .50, .75, .90, .99])
axes[1].axis("off")
axes[1].table(cellText=[[f"p{int(i*100)}", f"${v:,.0f}"] for i, v in qs.items()],
              colLabels=["percentil", "saldo"], loc="center", cellLoc="left")
axes[1].set_title("Percentiles de saldo")
plt.tight_layout(); plt.show()

print(f"Cuentas con saldo < $100:  {(dep.eom_balance < 100).mean():.1%}")
print(f"Cuentas con saldo == 0:    {(dep.eom_balance == 0).mean():.1%}")
print(f"\n→ El umbral `target.low_balance_abs` (hoy "
      f"${cfg.target['low_balance_abs']:,.0f}) debe caer en la cola izquierda,")
print("  no cortar clientes normales.")

In [ ]:
# ── 3. Mix de productos y de roles del CIF ──
rel = tables["relationships"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

dep.product_family.value_counts().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Cuentas-mes por familia de producto")

rc = rel.relationship_role.value_counts()
rc.plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Relaciones por rol del CIF")
plt.tight_layout(); plt.show()

share = (rc / rc.sum()).mul(100).round(1)
print("Peso de cada rol (%):")
print(share.to_string())
print(f"\n→ {share.get('joint', 0):.1f}% joint: si sumaras saldos sin ponderar por")
print("  ownership, inflarías los depósitos del banco en esa proporción.")

## 3 · Panel y target

Aquí se materializan las tres decisiones de diseño del modelo.

**Saldos ponderados por rol.** Una cuenta joint no cuenta entera para cada titular
(`1/n`), y un authorized signer no cuenta nada (peso `0`). Sin esto, los depósitos
del banco quedan inflados por doble conteo.

**Target compuesto.** No basta con el cierre formal: la attrition silenciosa
—saldo bajo **y** sin préstamos **y** sin actividad— sostenida durante
`target.persistence_months` cuenta igual. Es la que se escapa de los reportes.

**La espina es el maestro de clientes**, no las relaciones vigentes. Si se
construyera sobre relaciones, el cliente que cierra todo desaparecería del panel
justo en el mes que interesa, en vez de aparecer con ceros.

In [ ]:
state = build_cif_month_state(tables, cfg)
state_lab = label_attrition_state(state, cfg)

print(f"Estado CIF×mes: {state.shape[0]:,} filas × {state.shape[1]} columnas")
n = state_lab.cif_id.nunique()
ev_n = state_lab.dropna(subset=["churn_event_month"]).cif_id.nunique()
print(f"CIFs con evento de churn detectado: {ev_n:,} / {n:,} ({ev_n/n:.1%})")

In [ ]:
# Descomposición del target: ¿qué condición está mandando?
# El umbral de inactividad se deriva de config.yaml igual que en
# label_attrition_state, para que esta tabla no se desincronice del target.
inact_m = max(1, round(cfg.target["inactivity_days"] / 30))

conds = {
    "saldo bajo": state_lab.owned_balance < np.maximum(
        cfg.target["low_balance_abs"],
        state_lab.balance_baseline_12m.fillna(0) * cfg.target["low_balance_pct_of_baseline"]),
    "sin préstamos activos": state_lab.n_active_loans.fillna(0) <= 0,
    f"inactivo ≥{inact_m}m": state_lab.months_since_activity.fillna(0) >= inact_m,
    "sin relaciones vigentes": state_lab.n_relationships_any_role.fillna(0) <= 0,
    "ESTADO attrited (combinado)": state_lab.is_attrited_state,
}
pd.Series({k: f"{v.mean():.2%}" for k, v in conds.items()}, name="% de filas del panel").to_frame()

In [ ]:
# Se reusa el estado ya etiquetado de la celda anterior: reconstruirlo aquí
# duplicaría la parte más lenta del pipeline.
panel = build_panel(tables, cfg, state=state_lab)

# Estabilidad de la tasa base en el tiempo: si salta, hay un problema de datos
rate = panel.groupby("as_of_month").y.agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(rate.index, rate["mean"] * 100, marker="o", ms=4, color="crimson")
ax.axhline(panel.y.mean() * 100, ls="--", color="gray", lw=1,
           label=f"promedio {panel.y.mean():.2%}")
ax.set_title("Tasa de churn por mes de observación"); ax.set_ylabel("% churn")
ax.legend(); plt.tight_layout(); plt.show()

## 4 · Feature engineering

Para cada fila `(cif_id, as_of_month)`, **todo** se calcula sobre la ventana hacia
atrás `[t-11, t]`. Nunca sobre el futuro.

Nueve familias de señal:

| Familia | Qué captura |
|:---|:---|
| Nivel | El valor de hoy — saldo, productos, logins |
| Tendencia | Pendiente OLS sobre la ventana |
| Ratio a sí mismo | El cliente contra su propia línea base, no contra la media del banco |
| Drawdown | Caída desde el máximo de la ventana |
| Volatilidad | Dispersión de la serie |
| Aceleración | Cambio de la pendiente: si se desacelera o se desploma |
| Rachas | Meses consecutivos en una misma condición |
| Estacionalidad | Patrones recurrentes por mes del año |
| Relacionales | Contagio — el co-titular que ya se está yendo |

In [ ]:
import re

feats = build_features(panel, cfg, tables)
numeric, categorical = select_feature_columns(feats)
feats = clean_features(feats, numeric)

print(f"{len(numeric)} numéricas + {len(categorical)} categóricas")
print("\nMuestra por familia:")
for fam, pat in [("nivel",       r"^(owned_balance|txn_count_total|loan_balance|n_deposit_accounts)$"),
                 ("tendencia",   r"_slope_\d+m$"),
                 ("drawdown",    r"_drawdown_\d+m$"),
                 ("aceleración", r"_accel$"),
                 ("racha",       r"_declining_streak$|^months_since_"),
                 ("dominio",     r"dd_lost|loan_ending|cd_maturing"),
                 ("relacional",  r"co_owner|n_joint_accounts|n_signer_only")]:
    cols = [c for c in numeric if re.search(pat, c)]
    print(f"  {fam:14s} {len(cols):3d}  ej: {cols[:3]}")

## 5 · Split out-of-time con embargo

Dos protecciones contra leakage, en este orden:

1. **Corte temporal, no aleatorio.** Los últimos `split.oot_months` meses son el
   holdout. Un split aleatorio pondría el futuro dentro del train.
2. **Embargo.** Se eliminan del train las filas cuya ventana de etiqueta invade el
   periodo de test.

El embargo cuesta `gap + performance` meses de panel. Es caro y aun así no es
opcional: sin él, el holdout está contaminado y las métricas mienten hacia arriba.

In [ ]:
train, test = temporal_split(feats, cfg)

Xtr, ytr = train[numeric + categorical], train.y.to_numpy()
Xte, yte = test[numeric + categorical], test.y.to_numpy()
groups = train.cif_id.to_numpy()   # ← clave: el CV agrupa por esto

## 6 · Tuning con `StratifiedGroupKFold`

`groups=cif_id` es lo que impide que el mismo cliente esté a la vez en train y en
validación. Con un `StratifiedKFold` normal el PR-AUC sube varios puntos sin que el
modelo haya mejorado en nada.

La métrica de optimización es **average precision (PR-AUC)** — ni accuracy ni
ROC-AUC. Con una tasa base de 2-5%, predecir "no se va nadie" acierta el 96% de
las veces y no sirve para nada.

In [ ]:
%%time
searches = tune_all(Xtr, ytr, groups, numeric, categorical, cfg)

# `preds` guarda la probabilidad de cada candidato sobre el test. Es lo que
# permite más adelante puntuar exactamente el modelo que encabeza la tabla,
# incluidos los que no son un estimador de sklearn (GRU, blend).
results, fitted, preds = {}, {}, {}
for name, s in searches.items():
    preds[name] = s.best_estimator_.predict_proba(Xte)[:, 1]
    results[name] = evaluate(yte, preds[name])
    fitted[name] = s.best_estimator_

summarize(results)

## 7 · Stacking

Los `models.stack_top_k` mejores modelos por PR-AUC de CV entran como base learners,
con una regresión logística como meta-modelo sobre sus probabilidades out-of-fold.

In [ ]:
stack, chosen = build_stack(searches, cfg.models["stack_top_k"], cfg)
if stack is not None:
    print(f"Base learners: {chosen}")
    stack.fit(Xtr, ytr)
    preds["stacking"] = stack.predict_proba(Xte)[:, 1]
    results["stacking"] = evaluate(yte, preds["stacking"])
    fitted["stacking"] = stack
summarize(results)

## 8 · Red secuencial (GRU)

El GBM ve *resúmenes* de la trayectoria: pendientes, drawdowns, rachas. El GRU ve la
**serie mensual cruda** —`[muestras, meses, canales]`— y aprende la forma completa de
la caída.

Ahí está el valor real del ensemble: mezclar dos maneras distintas de mirar la misma
historia, no promediar cinco GBMs que se equivocan en los mismos clientes.

> **El early stopping y el peso de la mezcla se deciden en una partición de
> validación agrupada por CIF, sacada del train.** Usar el test para eso —elegir la
> época que mejor puntúa en el holdout y luego reportar ese holdout— infla las
> métricas por la misma puerta que el embargo cierra.

In [ ]:
%%time
from sklearn.model_selection import GroupShuffleSplit

sq = cfg.models["sequence_model"]

if not sq.get("enabled", True):
    print("ℹ️  Red secuencial desactivada en config.yaml "
          "(models.sequence_model.enabled = false) — se omite.")
else:
    win = cfg.panel["feature_window_months"]

    # Validación DENTRO del train, agrupada por CIF igual que el CV de los
    # modelos tabulares. Aquí se deciden el early stopping del GRU y el peso
    # del blend: hacerlo contra el test sería elegir hiperparámetros mirando
    # el holdout, que es justo lo que el embargo trata de impedir.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2,
                            random_state=cfg.split["random_state"])
    i_tr, i_val = next(gss.split(train, train.y, groups=train.cif_id))
    sub_tr, sub_val = train.iloc[i_tr], train.iloc[i_val]
    y_sub, y_val = sub_tr.y.to_numpy(), sub_val.y.to_numpy()

    # Las estadísticas de normalización salen solo del train y se reutilizan
    # en validación y test, para que un mismo saldo signifique lo mismo en los
    # tres tensores.
    Str, seq_stats = build_sequences(state, sub_tr, SEQUENCE_COLS, win, return_stats=True)
    Sval = build_sequences(state, sub_val, SEQUENCE_COLS, win, stats=seq_stats)
    Ste = build_sequences(state, test, SEQUENCE_COLS, win, stats=seq_stats)
    print(f"Tensor: {Str.shape}  [muestras, meses, canales]   "
          f"validación: {Sval.shape[0]:,} filas")

    gru = SequenceModel(n_features=Str.shape[2], hidden_size=sq["hidden_size"],
                        num_layers=sq["num_layers"], dropout=sq["dropout"],
                        lr=sq["lr"], epochs=sq["epochs"], batch_size=sq["batch_size"],
                        patience=sq["patience"]).fit(Str, y_sub, Sval, y_val)

    preds["gru_sequence"] = gru.predict_proba(Ste)[:, 1]
    results["gru_sequence"] = evaluate(yte, preds["gru_sequence"])

    # El tabular base se elige por su score de CV, no por el test; el peso de
    # la mezcla, en la partición de validación.
    best_tab = max(searches, key=lambda k: searches[k].best_score_)
    p_tab_val = fitted[best_tab].predict_proba(sub_val[numeric + categorical])[:, 1]
    p_gru_val = gru.predict_proba(Sval)[:, 1]
    w = max((0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
            key=lambda w: evaluate(y_val, (1 - w) * p_tab_val + w * p_gru_val)["pr_auc"])
    print(f"Base tabular: {best_tab}   peso del GRU elegido en validación: {w:.0%}")

    if w > 0:
        blend = f"blend_{best_tab}+gru_{w:.0%}"
        preds[blend] = (1 - w) * preds[best_tab] + w * preds["gru_sequence"]
        results[blend] = evaluate(yte, preds[blend])

summarize(results)

## 9 · Resultados

**Cómo leer la tabla: ignora `accuracy`.** Con una tasa base de 2-5% no dice nada.

| Métrica | Qué significa |
|:---|:---|
| `pr_auc` | Métrica principal — área bajo la curva precision-recall |
| `lift@10` | Cuántas veces mejor que aleatorio en el top 10% de riesgo |
| `capture@10` | Qué fracción de **todos** los churners cae en ese top 10% |
| `brier` | Qué tan bien calibradas están las probabilidades |

Retención solo puede llamar a una lista corta, así que `lift@10` y `capture@10` son
las que gobiernan la decisión de negocio.

In [ ]:
board = summarize(results)
best_name = board.index[0]
p_best = preds[best_name]          # exactamente el modelo que encabeza la tabla


# SHAP y contrafactuales necesitan un estimador de sklearn, y TreeExplainer
# solo es exacto sobre árboles. El GRU y el blend no lo son, y el stacking
# caería en el explainer genérico (muestreo, lento y no determinista). Así que
# la explicabilidad corre sobre el mejor GBM de la tabla, dicho explícitamente.
def _is_tree(est):
    final = est[-1] if hasattr(est, "named_steps") else est
    return any(k in type(final).__name__.lower()
               for k in ("forest", "lgbm", "xgb", "catboost", "boosting"))


explain_name = next((n for n in board.index if n in fitted and _is_tree(fitted[n])),
                    next(n for n in board.index if n in fitted))
explain_model = fitted[explain_name]
p_explain = preds[explain_name]

print(f"🏆 {best_name}: PR-AUC {board.loc[best_name,'pr_auc']:.4f}, "
      f"lift@10 {board.loc[best_name,'lift@10']:.1f}x")
if explain_name != best_name:
    print(f"   Explicabilidad y calibración sobre {explain_name} "
          f"(PR-AUC {board.loc[explain_name,'pr_auc']:.4f})")
board

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

dt = decile_table(yte, p_best)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].bar(dt.decil, dt.lift, color="steelblue")
axes[0].axhline(1, color="crimson", ls="--", lw=1)
axes[0].set_title("Lift por decil"); axes[0].set_xlabel("decil (1 = mayor riesgo)")

axes[1].plot(dt.decil * 10, dt["capture_%"], marker="o", color="darkorange")
axes[1].plot([0, 100], [0, 100], ls="--", color="gray", lw=1, label="aleatorio")
axes[1].set_title("Curva de captura"); axes[1].set_xlabel("% de la base contactada")
axes[1].set_ylabel("% de churners capturados"); axes[1].legend()

pr, rc, _ = precision_recall_curve(yte, p_best)
axes[2].plot(rc, pr, color="seagreen")
axes[2].axhline(yte.mean(), ls="--", color="gray", lw=1, label=f"base {yte.mean():.2%}")
axes[2].set_title("Curva Precision-Recall"); axes[2].set_xlabel("recall")
axes[2].set_ylabel("precision"); axes[2].legend()

plt.tight_layout(); plt.show()
dt

## 10 · Calibración

Necesaria en cuanto la probabilidad alimente un cálculo de valor en riesgo
(`P(churn) × saldo × margen`): ahí ya no basta con ordenar bien, el número tiene que
significar lo que dice. El stacking suele descalibrar.

In [ ]:
# El calibrador se ajusta en el primer tercio del test y se mide en el resto,
# así nunca se evalúa sobre filas que ya vio. Como el panel viene ordenado
# por CIF, el corte cae entre clientes: solo el que queda justo en el límite
# aparece en ambos tramos.
n_cal = len(Xte) // 3
cal = calibrate(explain_model, Xte.iloc[:n_cal], yte[:n_cal])

y_ev = yte[n_cal:]
p_raw = p_explain[n_cal:]                              # mismo modelo, sin calibrar
p_cal = cal.predict_proba(Xte.iloc[n_cal:])[:, 1]

before, after = calibration_table(y_ev, p_raw), calibration_table(y_ev, p_cal)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(before.prob_predicha, before.tasa_observada, "o-", label="sin calibrar")
ax.plot(after.prob_predicha, after.tasa_observada, "s-", label="isotónica")
lim = max(before.prob_predicha.max(), after.prob_predicha.max()) * 1.1
ax.plot([0, lim], [0, lim], "--", color="gray", lw=1, label="perfecta")
ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("tasa observada")
ax.set_title(f"Curva de calibración — {explain_name}"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Brier: {evaluate(y_ev, p_raw)['brier']:.5f} → "
      f"{evaluate(y_ev, p_cal)['brier']:.5f}")

## 11 · Explicabilidad (SHAP)

`TreeExplainer` es exacto y determinista sobre GBMs. Por eso esta sección corre
sobre el mejor modelo **de árboles** de la tabla, que no siempre es el que la
encabeza: sobre un stacking o un blend habría que caer al explainer genérico, que
muestrea y deja de ser reproducible. La celda anterior imprime cuál se usó.

**Por qué no LIME:** reajusta un modelo lineal local con muestreo aleatorio, así que
dos corridas sobre el mismo cliente devuelven explicaciones distintas. Inaceptable si
la explicación puede terminar en un expediente de model risk (SR 11-7).

`sanity_check` avisa si aparece una variable geográfica o un identificador en el
top-10: casi siempre es proxy de algo, o leakage.

In [ ]:
sv, Xt = shap_analysis(explain_model, Xte, cfg)
imp = global_importance(sv, Xt, top_n=25)

fig, ax = plt.subplots(figsize=(9, 7))
top = imp.head(20).iloc[::-1]
colors = ["crimson" if d > 0.05 else "steelblue" if d < -0.05 else "gray"
          for d in top.direccion]
ax.barh(top.feature, top.shap_abs_mean, color=colors)
ax.set_xlabel("|SHAP| medio"); ax.set_title(f"Importancia global — {explain_name} (rojo = sube riesgo)")
plt.tight_layout(); plt.show()

_ = sanity_check(imp, expected_top=["direct_deposit", "balance", "drawdown", "slope"])
imp

In [ ]:
# Explicación local: un cliente concreto del top de riesgo.
# Se ordena por p_explain (el modelo que produjo el SHAP), no por p_best, para
# que el cliente señalado y la explicación vengan del mismo modelo.
top_idx = Xte.index[np.argsort(p_explain)[::-1][:1]][0]

if top_idx in Xt.index:
    display(explain_customer(explain_model, Xte, top_idx, top_n=12,
                             shap_values=sv, Xt=Xt))
else:
    # No cayó en la muestra de `shap_sample_size`. TreeExplainer es exacto fila
    # a fila, así que se calcula solo para esta en vez de renunciar: si no, la
    # explicación local desaparece cada vez que el sorteo no lo incluye.
    display(explain_customer(explain_model, Xte, top_idx, top_n=12))

## 12 · Contrafactuales accionables

SHAP dice *qué pesó*. Retención necesita *qué hacer*. Esta celda simula la
intervención sobre los clientes de mayor riesgo —moviendo las variables de
`explain.counterfactual_features`— y mide cuánto baja la probabilidad.

In [ ]:
cf = counterfactual_analysis(explain_model, Xte, cfg)
cf

## 13 · Lista de scoring y guardado

Lo que se entrega al equipo de retención: el top de riesgo con su probabilidad
**calibrada** y el saldo en riesgo asociado. Sale del tramo de test que el calibrador
no llegó a ver, para que el número sea comparable entre clientes y sumable.

Se guardan además el leaderboard, la importancia SHAP global y los dos modelos —el
crudo y el calibrado—.

In [ ]:
import joblib

# La lista sale del tramo de test que el calibrador no vio, y con la
# probabilidad ya calibrada: sin eso, P(churn) × saldo no es comparable
# entre clientes ni sumable como valor en riesgo.
scored = test.iloc[n_cal:][["cif_id", "as_of_month", "owned_balance", "y"]].copy()
scored["p_churn"] = p_cal
scored["valor_en_riesgo"] = scored.p_churn * scored.owned_balance
scored = scored.sort_values("p_churn", ascending=False)

out = cfg.artifacts_path()
scored.head(500).to_csv(out / "lista_retencion_top500.csv", index=False)
board.to_csv(out / "leaderboard.csv")
imp.to_csv(out / "shap_importance.csv", index=False)
joblib.dump(explain_model, out / "modelo_final.joblib")
joblib.dump(cal, out / "modelo_calibrado.joblib")

print(f"💾 Guardado en {out}  (modelo: {explain_name})")
print(f"\nValor total en riesgo (top 10%): "
      f"${scored.head(len(scored)//10).valor_en_riesgo.sum():,.0f}")
scored.head(15)

---

## Siguiente paso

1. Cambia `cfg.data["source"] = "drive"` y sube los extractos del agente SQL.
2. Revisa la EDA de **saldos** y ajusta `target.low_balance_abs` en `config.yaml` a lo
   que digan tus datos — no lo inventes.
3. Revisa el gráfico de **aperturas/cierres**: si hay un pico, corta la historia ahí.
4. Corre la descomposición del target. Si una sola condición domina, la definición
   está desbalanceada.
5. Antes de presentar: valida el SHAP contra intuición de negocio y corre un análisis
   de proxies sobre `state` y `preferred_language` (Fair Lending / UDAAP).